# **LLM을 이해하는 세 단어 : 토큰 · 확률 · 파라미터**

---

## 용어 정의

| # | 단어 | 슬라이드 한 줄 | 이 노트북에서 직접 볼 것 |
|---|---|---|---|
| 1 | **토큰 (Token)** | AI가 글을 다루는 '조각' 단위 | 내 문장이 실제로 몇 조각으로 쪼개지는지, 한글이 왜 더 비싼지 |
| 2 | **확률 (Probability)** | 다음 토큰을 '확률로' 선택 | AI가 다음 단어를 고를 때 머릿속 후보 순위표 |
| 3 | **파라미터 (Parameter)** | 학습으로 조정된 '손잡이' | 손잡이가 실제 몇 개인지 세어보고, 손잡이를 돌려 성격을 바꿔보기 |


---

## 코드 실행 방법

1. 이 노트북을 [Google Colab](https://colab.research.google.com) 에 업로드합니다. (`파일 → 노트북 업로드`)
2. 상단 메뉴 **런타임 → 모두 실행** 을 누릅니다.
3. 첫 실행은 모델을 내려받느라 **2~3분** 걸립니다. 두 번째부터는 즉시 실행됩니다.

---
# **0. 준비 (한 번만 실행)**

Colab에는 이미 대부분의 라이브러리라 설치되어 있습니다. 없는 것만 설치 진행합니다.

In [ ]:
# [0-1] 필요한 라이브러리 설치
!pip install -q transformers tiktoken 2>/dev/null

import warnings, os
warnings.filterwarnings("ignore")          # 실습에 방해되는 경고 메시지 숨기기
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print("PyTorch  :", torch.__version__)
print("실행 장치 :", "GPU (빠름)" if torch.cuda.is_available() else "CPU (느리지만 충분합니다)")

In [ ]:
# [0-2] 이 노트북 전체에서 쓸 '텍스트 막대그래프' 도우미 함수
#       (글자로 막대를 그립니다)

import unicodedata

def pad(s, n):
    '''한글은 화면에서 두 칸을 차지하므로, 실제 폭을 계산해 줄을 맞춘다'''
    w = sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in str(s))
    return str(s) + " " * max(0, n - w)

def bar(p, width=34):
    '''확률 p(0~1)를 ■ 막대로 그려 문자열로 돌려준다'''
    filled = int(round(p * width))
    return "■" * filled + "·" * (width - filled)

def show_ranking(pairs, title="", width=34):
    '''[(이름, 확률), ...] 목록을 순위표로 출력'''
    if title:
        print(title)
        print("─" * (width + 26))
    for i, (name, p) in enumerate(pairs, 1):
        label = repr(name)[1:-1]                      # 공백·줄바꿈이 보이도록
        print(f"{i}위 {pad(label, 14)} {bar(p, width)} {p*100:6.2f}%")
    print()

# 동작 확인
show_ranking([("서울", 0.82), ("부산", 0.07), ("대구", 0.03)], "예시 출력")

---
# **1. 토큰 (Token) — AI가 글을 다루는 '조각' 단위**


## 1-1. 발전 과정

| 시기 | 사건 | 의미 |
|---|---|---|
| 1994 | Philip Gage, **BPE(Byte Pair Encoding)** 압축 알고리즘 발표 | 자주 붙어 나오는 글자쌍을 하나로 묶는 아이디어 |
| 2016 | Sennrich 외, BPE를 기계번역에 적용 (ACL 2016) | "모르는 단어" 문제를 조각내기로 해결 |
| 2019~ | GPT-2 이후 거의 모든 LLM이 이 방식 채택 | 오늘날 토큰의 표준 |

> 1. BPE 예:aaabdaaabac 라는 문자열이 있다면, 가장 흔한 쌍은 aa. 이걸 Z로 바꾸면 ZabdZabac. 다음으로 흔한 쌍 ab를 Y로 바꾸면 ZYdZYac... 이런 식으로 문자열이 점점 짧아집니다.
> 2. 더는 압축 이득이 없을 때까지 반복합니다.
> 3. 2016년 논문은 이 절차를 그대로 가져오되, **압축**이 아니라 **어휘 사전 만들기**로 목적을 바꿨습니다.
- **핵심**: AI는 글자(음절)도, 단어도 아닌 **'자주 쓰이는 덩어리'** 단위로 글을 봅니다.
- 이 덩어리 하나가 **토큰 1개**이고, ChatGPT·Claude의 **요금과 길이 제한이 전부 이 토큰 개수로 계산**됩니다.

> **한 줄 정리** — 토큰은 AI의 '글자 수 세는 방법'이자, 곧 **요금 단위**입니다.

## 1-2. 어휘 사전과 토큰

In [ ]:
import tiktoken

# 사용 가능한 모든 인코딩 이름 목록을 가져옵니다.
available_encodings = tiktoken.list_encoding_names()

print("사용 가능한 tiktoken 인코딩 목록(토큰 사전): ")
for encoding_name in available_encodings:
    print(f"- {encoding_name}")

### 1️⃣ 내 문장은 몇 조각으로 쪼개질까?

In [ ]:
# 1. 내 문장은 몇 조각으로 쪼개질까?
import tiktoken

enc = tiktoken.get_encoding("o200k_base")   # GPT-4o / GPT-5 계열이 쓰는 토큰 사전

def to_pieces(text):
    '''문장을 토큰 조각 목록으로 변환 (한글 반쪽 토큰도 안전하게 표시)'''
    ids = enc.encode(text)
    pieces = [enc.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in ids]
    return ids, pieces

문장 = "안녕하세요, 오늘 강의 재미있게 들으세요!"

ids, pieces = to_pieces(문장)
print("원문     :", 문장)
print("토큰 개수 :", len(ids), "개")
print("쪼갠 모습 :", " / ".join(pieces))

### 2️⃣ 한글 vs 영어 — 같은 뜻인데 토큰 수가 다르다
`안녕하세요` 한 단어가 여러 조각으로 쪼개지고, 조각 하나가 글자 하나와 **일치하지 않는다**는 점.
슬라이드의 *"AI는 글자가 아니라 이 '토큰'을 셉니다"* 가 바로 이 장면입니다.

In [ ]:
# 2. 한글 vs 영어 — 같은 뜻인데 토큰 수가 다르다
쌍 = [
    ("Hello",                         "안녕하세요"),
    ("Thank you",                     "감사합니다"),
    ("Artificial Intelligence",       "인공지능"),
    ("The weather is nice today.",    "오늘 날씨가 좋습니다."),
    ("Please summarize this document.", "이 문서를 요약해 주세요."),
]

print(f"{pad('영어', 34)}{'토큰':>4}   |  {pad('한국어', 26)}{'토큰':>4}   배율")
print("─" * 84)
합계_en = 합계_ko = 0
for en, ko in 쌍:
    n_en, n_ko = len(enc.encode(en)), len(enc.encode(ko))
    합계_en += n_en; 합계_ko += n_ko
    print(f"{pad(en, 34)}{n_en:>4}   |  {pad(ko, 26)}{n_ko:>4}   {n_ko/n_en:>4.1f}배")
print("─" * 84)
print(f"{pad('합계', 34)}{합계_en:>4}   |  {pad('합계', 26)}{합계_ko:>4}   {합계_ko/합계_en:>4.1f}배")

### 3️⃣ 그래서 요금이 얼마나 차이 날까

In [ ]:
# 3. 그래서 요금이 얼마나 차이 날까? (토큰 단가는 예시값입니다)
단가_100만토큰당_달러 = 1.25           # ※ 예시 수치 — 실제 요금은 모델·시점마다 다릅니다
환율 = 1412                            # ※ 예시 수치

원고 = "이 문서를 세 문장으로 요약해 주세요."
반복 = 10000                           # 이 요청을 1만 번 보낸다고 가정

n_ko = len(enc.encode(원고)) * 반복
n_en = len(enc.encode("Please summarize this document in three sentences.")) * 반복

for 이름, n in [("한국어", n_ko), ("영어", n_en)]:
    비용 = n / 1_000_000 * 단가_100만토큰당_달러 * 환율
    print(f"{pad(이름, 8)}{n:>10,} 토큰   →   약 {비용:,.0f}원")

print("\n※ 단가·환율은 설명용 예시값입니다. 실제 요금은 각 서비스의 공식 요금표를 확인하세요.")

### 4️⃣ AI 눈에는 이렇게 보입니다

In [ ]:
# 4. AI 눈에는 이렇게 보입니다
실험 = [
    "커피",
    "커  피",                      # 띄어쓰기 넣으면?
    "coffee",
    "COFFEE",                     # 대문자로 바꾸면?
    "ㅋㅋㅋㅋㅋㅋ",
    "1234567890",
    "https://www.example.com/very/long/path",
    "자문화중심주의",              # 흔치 않은 긴 단어
]

for t in 실험:
    ids, pieces = to_pieces(t)
    print(f"{pad(t, 42)}{len(ids):>2}토큰   {' / '.join(pieces)}")

### 5️⃣ **질문?**

| 질문 |  대답 |
|---|---|
| "`커피`와 `커 피`, 토큰 수가 왜 다를까요?" |  ?               |
| "`COFFEE`가 `coffee`보다 토큰이 많은 이유는?" | ?   |
| "그럼 프롬프트를 영어로 쓰면 이득일까요?" |  ?  |

> **한 줄 정리** — 한글은 영어보다 대략 **2~3배** 토큰을 씁니다. 같은 말을 해도 요금이 더 나온다는 뜻입니다.

## **[미션] 내 이름의 몸값(토큰량) 재기**
**자신의 이름과 자기소개 한 줄**을 넣고 다음 내용을 확인해 보세요.

- 몇 토큰인가요?
- 같은 내용을 영어로 쓰면 몇 토큰인가요?

---
# **2. 확률 (Probability) — 다음 토큰을 '확률로' 선택**



## 2-1. 발전 과정

| 시기 | 사건 | 의미 |
|---|---|---|
| 1948 | Claude Shannon, 정보이론 논문에서 **영어를 확률로 예측**하는 실험 | "다음 글자 맞히기"의 원조 |
| 1980~90년대 | n-gram 언어모델 (앞의 2~3단어만 보고 다음 단어 예측) | 통계적 언어모델의 시작 |
| 2017~ | Transformer 등장 → 문맥 전체를 보고 다음 토큰의 확률 계산 | 오늘날 LLM |

- **핵심**: AI는 답을 '아는' 게 아니라, 후보 토큰 수만~수십만 개 전부에 **점수를 매기고 그중에서 고릅니다.**
- 그래서 같은 질문에도 답이 매번 조금씩 다릅니다.

> **한 줄 정리** — LLM은 검색이 아니라 **"다음 조각 맞히기 게임"** 을 계속 이어가는 기계입니다.

### ⏺️ 오늘날 LLM 원리의 원조
1948년은 "확률표로 글을 만들어봤다", 1951년은 "사람이 얼마나 잘 맞히는지 재봤다". 둘 다 "**다음 글자는 확률로 결정된다**"는 오늘날 LLM 원리의 원조 예제

In [ ]:
# Shannon(1948) 영어 근사 + Shannon(1951) 글자 맞히기 게임 재현
# 원본은 방대한 영어 말뭉치를 손으로 세었지만, 여기선 원리 확인용 소형 예시 텍스트 사용
import random
from collections import defaultdict, Counter

random.seed(42)

말뭉치 = """
the quick brown fox jumps over the lazy dog the dog barks at the fox
the fox runs into the forest the forest is dark and quiet a bird sings
in the tree the sun sets behind the hill the sky turns orange and red
""".lower()
text = "".join(c for c in 말뭉치 if c.isalpha() or c == " ")


# ── 1948: 차수를 높여가며 가짜 영어 생성 ──────────────────
# 주어진 '차수(order)'만큼의 앞선 글자를 문맥(context)으로 삼아,
# 그 다음에 어떤 글자가 얼마나 자주 나타났는지를 세어 'n-그램 모델'을 만듭니다.
# 예를 들어, 'th' 다음에 'e'가 얼마나 많이 왔는지 등을 기록합니다.
def build_ngram(order):
    model = defaultdict(Counter)
    for i in range(len(text) - order):
        ctx, nxt = text[i:i+order], text[i+order]
        model[ctx][nxt] += 1
    return model


# 이 모델을 사용해 가짜 영어를 생성합니다. 문맥 길이(차수)를 늘릴수록 생성되는
# 문장이 원래 영어와 더 비슷해지는 것을 보여주는데, 이는 AI가 더 많은 문맥을
# 볼수록 예측 정확도가 높아진다는 원리를 시각적으로 나타냅니다
def ngram_generate(order, n):
    model = build_ngram(order)
    ctx = text[:order]
    out = [ctx]
    for _ in range(n - order):
        choices = model.get(ctx) or Counter(text)
        pop, w = zip(*choices.items())
        nxt = random.choices(pop, weights=w, k=1)[0]
        out.append(nxt)
        ctx = (ctx + nxt)[-order:]
    return "".join(out)

print("=== 문맥 길이(차수)를 늘릴수록 그럴듯해지는 가짜 영어 ===")
for order in [1, 2, 4]:
    print(f"직전 {order}글자만 봄 :", ngram_generate(order, 60))


# ── 1951: 다음 글자 맞히기 게임 ────────────────────────
# 섀넌의 1951년 '글자 맞히기 게임'을 재현합니다. 주어진 문장에서 각 글자를 예측하기
# 위해 n-그램 모델이 몇 번째 시도 만에 정답을 맞히는지 측정합니다. 평균 시도 횟수가
# 낮을수록(1에 가까울수록) 해당 문장은 예측하기 쉬운, 즉 정보량이 적은 문장이라고
# 해석할 수 있습니다
def guessing_game(sentence, order=3):
    model = build_ngram(order)
    total = 0
    for i in range(order, len(sentence)):
        ctx, actual = sentence[i-order:i], sentence[i]
        ranked = [c for c, _ in (model.get(ctx) or Counter(text)).most_common()]
        guess_no = ranked.index(actual) + 1 if actual in ranked else len(ranked) + 1
        total += guess_no
        print(f"  문맥 '...{ctx}' 다음 → 정답 '{actual}'   {guess_no}번째 시도에 맞힘")
    print(f"\n평균 시도 횟수: {total/(len(sentence)-order):.2f}회 (1에 가까울수록 예측하기 쉬운 문장)")

print("\n=== 사람 대신 통계 모델로 재현한 '글자 맞히기 게임' ===")
guessing_game("the fox runs into the forest", order=3)


# 이 코드는 현대 대규모 언어 모델(LLM)이 '다음 토큰(글자나 단어 조각) 맞히기'를 통해
# 문장을 생성하는 기본 원리를 단순화된 형태로 보여주는 중요한 예시입니다.


## 2-2. 언어모델 사용하여 다음 단어 확률 확인하기

### 1️⃣ 작은 언어모델 불러오기

In [ ]:
# 1. 작은 언어모델 불러오기 (약 0.5B 크기 — Colab 무료에서도 가볍게 돌아갑니다)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, torch.nn.functional as F

MODEL_NAME = "Qwen/Qwen2.5-0.5B"       # 한국어·영어를 모두 조금씩 아는 작은 모델
# 대안: "Qwen/Qwen2.5-1.5B" (더 똑똑하지만 무거움) / "skt/kogpt2-base-v2" (한국어 전용·초경량)

print("모델 내려받는 중… 처음 한 번만 2~3분 걸립니다")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("준비 완료! 실행 장치:", device)

### 2️⃣ 다음 토큰 후보 순위 뽑는 함수

In [ ]:
# 2."다음 토큰 후보 순위표"를 뽑는 함수
@torch.no_grad()
def next_token_ranking(prompt, k=8, temperature=1.0):
    '''prompt 다음에 올 토큰 후보 상위 k개와 확률을 돌려준다'''
    temperature = max(float(temperature), 0.01)       # 0으로 나누기 방지
    ids = tok(prompt, return_tensors="pt").to(device)
    logits = model(**ids).logits[0, -1]          # 마지막 위치의 '점수표' (어휘 전체)
    probs = F.softmax(logits / temperature, dim=-1)   # 점수표 → 확률로 변환
    top = torch.topk(probs, k)
    return [(tok.decode([i]), float(p)) for p, i in zip(top.values, top.indices)]

# 실행!
프롬프트 = "대한민국의 수도는"
show_ranking(next_token_ranking(프롬프트), f'입력: "{프롬프트}"  →  AI가 고민 중인 다음 조각 Top 8')

### 3️⃣ 영어로도 확인 — 작은 모델일수록 영어에서 더 또렷함

In [ ]:
# 3. 영어로도 확인 — 작은 모델일수록 영어에서 더 또렷합니다
for p in ["The capital of France is",
          "1 + 1 =",
          "Once upon a time, there was a"]:
    show_ranking(next_token_ranking(p, k=5), f'입력: "{p}"')

### 4️⃣ 한 조각씩 이어붙이기 — AI가 문장을 만드는 과정

In [ ]:
# 4. 한 조각씩 이어붙이기 — AI가 문장을 만드는 과정을 슬로우모션으로
@torch.no_grad()
def slow_motion(prompt, steps=10):
    현재 = prompt
    print(f'시작: "{prompt}"\n')
    for step in range(1, steps + 1):
        후보 = next_token_ranking(현재, k=3)
        선택, 확률 = 후보[0]                    # 항상 1등을 고르는 방식(= greedy)
        경쟁 = "   ".join(f"{repr(n)[1:-1]}({p*100:.0f}%)" for n, p in 후보[1:])
        print(f"{step:2}단계  선택 → {pad(repr(선택)[1:-1], 12)}{확률*100:5.1f}%    (경쟁 후보: {경쟁})")
        현재 += 선택
    print(f'\n완성: "{현재}"')

slow_motion("대한민국의 수도는", steps=10)

### 5️⃣ 그래서 답이 매번 달라집니다
AI는 문장을 **통째로 떠올리는 게 아니라 한 조각씩** 이어붙입니다.
매 단계마다 2등·3등 후보가 바짝 붙어 있는 걸 보세요. **그중 하나를 뽑는 순간마다 다른 이야기가 시작됩니다.**

In [ ]:
# [2-5] 그래서 답이 매번 달라집니다 — 같은 질문 5번
prompt = "오늘 점심으로 먹기 좋은 음식은"
ids = tok(prompt, return_tensors="pt").to(device)

print(f'질문: "{prompt}"\n')
for i in range(1, 6):
    out = model.generate(**ids, max_new_tokens=20, do_sample=True, temperature=1.0,
                         top_p=0.9, pad_token_id=tok.eos_token_id)
    print(f"{i}번째: {tok.decode(out[0], skip_special_tokens=True)}")
    print()

---
# **3. 파라미터 (Parameter) — 학습으로 조정된 '손잡이'**

## 용어 정의

| 구분 | 무엇인가 | 개수 | 누가 정하나 | 우리가 만질 수 있나 |
|---|---|---|---|---|
| **① 학습 파라미터 (가중치)** | 모델 안에 저장된 숫자값. AI의 '실력' 그 자체 | 수억 ~ 수조 개 | 학습 과정이 자동으로 | **아니오** (고정) |
| **② 생성 파라미터 (설정값)** | 온도(temperature) 등 답변 방식 조절 스위치 | 5~10개 | 사용자가 | **예** |

> - 강의자료에서 언급한 *"수십억~수조 개의 미세한 손잡이 값"* 은 **①번**을 말합니다.
> - 그런데 실제로 우리가 **돌려볼 수 있는 손잡이는 ②번**이죠. 아래에서 둘 다 만져봅니다.


## 모델 예

| 모델 | 발표 | 파라미터 수 |
|---|---|---|
| GPT-2 | OpenAI, 2019 | 15억 (1.5B) |
| GPT-3 | OpenAI, 2020 (논문 *Language Models are Few-Shot Learners*) | **1,750억 (175B)** |
| 최신 대형 모델 | 2024~ | 공식 미공개가 대부분 — **알 수 없습니다** |

> ※ 최신 상용 모델(GPT-5, Claude, Gemini 등)의 정확한 파라미터 수는 **공개되지 않았습니다.** 인터넷에 도는 수치는 대부분 추정치입니다.

## 3-1. 모델 파라미터 확인
앞에서 사용한 모델 : "Qwen/Qwen2.5-0.5B"

### 1️⃣ 학습 파라미터

In [ ]:
# 1. ① 학습 파라미터 — 이 모델의 '손잡이'는 실제로 몇 개일까?
총합 = sum(p.numel() for p in model.parameters())

print(f"이 작은 모델의 파라미터 수 : {총합:,} 개")
print(f"                          = 약 {총합/1e8:.1f}억 개")
print(f"파일로 저장하면 (숫자 1개당 4바이트) : 약 {총합*4/1e9:.2f} GB\n")

print("어디에 몇 개나 들어있나 (상위 5개 부품)")
print("─" * 62)
크기순 = sorted(((n, p.numel()) for n, p in model.named_parameters()),
               key=lambda x: -x[1])[:5]
for n, c in 크기순:
    print(f"{n:<44} {c:>12,}")

In [ ]:
# 전체 모델 구조 출력
# print(model)

# # 더 간결한 요약
# for name, module in model.named_modules():
#     if 'mlp' in name.lower():
#         print(f"{name}: {module}")

### 2️⃣ 모델의 크기 느껴보기

In [ ]:
# 2. 감이 안 오시죠? 크기를 몸으로 느껴보기
비교 = [
    ("지금 이 모델",              총합),
    ("GPT-2 (2019)",              1_500_000_000),
    ("GPT-3 (2020)",              175_000_000_000),
    ("사람 뇌의 시냅스 (추정)",    100_000_000_000_000),
]
최대 = max(v for _, v in 비교)

print(f"{'':<28}{'파라미터 / 시냅스 수':>22}")
print("─" * 86)
for 이름, v in 비교:
    print(f"{pad(이름, 28)}{v:>20,}  {bar(v/최대, 30)}")

print(f"\n※ GPT-2·GPT-3 수치는 OpenAI 공식 논문 기준입니다.")
print(f"※ 뇌 시냅스 수는 신경과학계의 통상적 추정치이며 연구마다 다릅니다. (추측성 비교입니다)")
print(f"\n비유: 지금 이 모델이 '메모지 한 장'이라면, GPT-3는 그 메모지 {175_000_000_000/총합:,.0f}장 분량입니다.")

### 3️⃣ 파리미터 돌려보기 - 온도(temperature)

In [ ]:
# 3. ② 생성 파라미터 '온도(temperature)' — 손잡이를 직접 돌려봅시다
#       온도가 낮으면: 1등만 고른다 (안전·반복적)
#       온도가 높으면: 하위 후보에도 기회를 준다 (창의적·엉뚱)

프롬프트 = "The capital of France is"

for T in [0.2, 1.0, 2.0]:
    라벨 = {0.2: "낮음 — 모범생 모드", 1.0: "기본 — 균형", 2.0: "높음 — 자유분방 모드"}[T]
    show_ranking(next_token_ranking(프롬프트, k=5, temperature=T),
                 f"온도 {T}  ({라벨})")

### 4️⃣ 온도에 따라 실제 문장이 어떻게 달라지나 확인하기
- **후보 목록은 그대로인데 막대 길이만 바뀝니다.**
- 온도는 AI에게 새 지식을 주는 게 아니라, **"1등을 얼마나 밀어줄지"** 를 정하는 손잡이일 뿐입니다.

In [ ]:
# 4. 온도에 따라 실제 문장이 어떻게 달라지나
프롬프트 = "인공지능은"
ids = tok(프롬프트, return_tensors="pt").to(device)

for T, 설명 in [(0.2, "안전하지만 지루함"), (0.8, "적당한 균형"), (1.6, "창의적이지만 헛소리 위험")]:
    print(f"── 온도 {T} : {설명} " + "─" * 30)
    for i in range(2):
        out = model.generate(**ids, max_new_tokens=30, do_sample=True, temperature=T,
                             top_p=0.95, pad_token_id=tok.eos_token_id)
        print("  ", tok.decode(out[0], skip_special_tokens=True).replace("\n", " "))
    print()

### 5️⃣ 실무에서 자주 쓰는 손잡이 정리

In [ ]:
# 5. 실무에서 자주 쓰는 손잡이 정리 (참고용 표)
표 = [
    ("temperature", "0.0 ~ 2.0", "0.7", "높을수록 창의적/불안정, 낮을수록 일관/반복"),
    ("top_p",       "0.0 ~ 1.0", "0.9", "확률 상위 90%까지만 후보로 고려 (엉뚱한 답 차단)"),
    ("max_tokens",  "1 ~ 수만",  "-",   "답변 길이 상한. 요금과 직결"),
    ("seed",        "정수",       "-",   "지정하면 같은 답 재현 가능 (지원하는 서비스만)"),
]
print(f"{pad('손잡이', 16)}{pad('범위', 14)}{pad('무난한 값', 12)}설명")
print("─" * 92)
for a, b, c, d in 표:
    print(f"{pad(a, 16)}{pad(b, 14)}{pad(c, 12)}{d}")

print("\n[★ 상황별 추천]")
추천 = [("사실 확인·번역·요약", "0.0 ~ 0.3"), ("일상 대화·기획 회의", "0.7 ~ 0.9"),
        ("아이디어 발상·카피 문구", "1.0 ~ 1.3")]
for 상황, 값 in 추천:
    print(f"  · {pad(상황, 26)}temperature {값}")

## **[미션] 온도 실험 보고서**
`프롬프트`를 **"내가 오늘 기분이 좋은 이유는"** 으로 바꾸고,
온도를 `0.2 / 0.8 / 1.6` 으로 각각 3번씩 돌려보세요.

| 온도 | 문장이 어땠나 (한 줄 메모) | 어떤 용도에 쓰면 좋을까 |
|---|---|---|
| 0.2 | | |
| 0.8 | | |
| 1.6 | | |

---
# **4. 종합**

> **토큰으로 쪼개고, 확률로 고르고, 파라미터로 판단한다 — LLM 동작의 전부입니다.**



In [ ]:
# 입력 한 줄이 답변이 되기까지, 전 과정 한눈에 보기
def 전과정_보기(문장, temperature=0.8, steps=12):
    print("=" * 76)
    print(f'입력 : "{문장}"')
    print("=" * 76)

    # ① 토큰 — 쪼개기
    ids_list = tok(문장)["input_ids"]
    조각 = [tok.decode([i]) for i in ids_list]
    print(f"\n① 토큰   {len(조각)}조각으로 쪼갬")
    print("        " + " / ".join(repr(p)[1:-1] for p in 조각))

    # ② 확률 — 다음 조각 후보 순위
    print(f"\n② 확률   다음 조각 후보 (온도 {temperature} 적용)")
    for 이름, p in next_token_ranking(문장, k=5, temperature=temperature):
        print(f"        {pad(repr(이름)[1:-1], 14)}{bar(p, 26)} {p*100:5.1f}%")

    # ③ 파라미터 — 손잡이를 돌려 최종 생성
    print(f"\n③ 파라미터  {총합:,}개의 학습 손잡이 + 온도 {temperature}로 최종 생성")
    ids = tok(문장, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=steps, do_sample=True,
                         temperature=temperature, top_p=0.9,
                         pad_token_id=tok.eos_token_id)
    print(f'\n결과 : "{tok.decode(out[0], skip_special_tokens=True)}"')
    print("=" * 76 + "\n")


전과정_보기("아침에 마시는 커피는")

## **[실습] 토큰으로 쪼개고, 확률로 고르고, 파라미터로 판단한다**

In [ ]:
# 여기를 바꿔가며 자유롭게 실험해 보세요
내_문장 = "학점을 잘 받기 위해 내가 할 수 있는 것은"    # ← 이 줄만 바꾸면 됩니다
내_온도 = 0.9                                    # ← 0.2 / 0.9 / 1.6 으로 바꿔보세요

전과정_보기(내_문장, temperature=내_온도)